In [ ]:
%config Completer.use_jedi = False
%load_ext autoreload

# Imports and Loads

In [ ]:
from crawler.src import crawler 
from ast import literal_eval

In [ ]:
import importlib
importlib.reload(crawler)

In [ ]:
import pandas as pd
from scipy.sparse import coo_matrix, csr_matrix
import numpy as np
from sklearn.preprocessing import LabelEncoder
from implicit.als import AlternatingLeastSquares
import csv
import requests
from bs4 import BeautifulSoup

In [ ]:
pd.options.display.max_columns=200
pd.options.display.max_rows=200

# Functions

In [ ]:
# from bs4 import BeautifulSoup
# import bs4
# import requests
import pandas as pd
# import json
# from requests.compat import urljoin
# from datetime import datetime
# import re
# from time import sleep
# import numpy as np
# from zipfile import ZipFile
# from io import BytesIO
# import math


# from selenium import webdriver
# from selenium.webdriver.chrome.options import Options
# from selenium.webdriver.common.by import By
# from selenium.webdriver.support.ui import WebDriverWait
# from selenium.webdriver.support import expected_conditions as EC
# from selenium.webdriver.chrome.service import Service

In [ ]:
def get_driver_and_cookies():
    LOGIN_USERNAME_FIELD = '//*[@id="inputUsername"]'
    LOGIN_PASSWORD_FIELD = '//*[@id="inputPassword"]'
    LOGIN_BUTTON = '//*[@id="mainbody"]/div/div/gg-login-page/div[1]/div/gg-login-form/form/fieldset/div[3]/button[1]'

    with open("/home/msnow/config.json", "r") as fp:
        secrets = json.load(fp)
    USERNAME = secrets["bgg_crawler"]["username"]
    PASSWORD = secrets["bgg_crawler"]["password"]

    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    cookies = {}

    driver = webdriver.Chrome(
        service=Service("/usr/lib/chromium-browser/chromedriver"),
        options=chrome_options,
    )
    driver.get("https://boardgamegeek.com/login")
    login = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, LOGIN_USERNAME_FIELD))
    )
    password = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, LOGIN_PASSWORD_FIELD))
    )

    login_button = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, LOGIN_BUTTON))
    )

    login.send_keys(USERNAME)
    password.send_keys(PASSWORD)

    login_button.click()
    sleep(1)
    selenium_cookies = driver.get_cookies()
    for cookie in selenium_cookies:
        cookies[cookie["name"]] = cookie["value"]
    return cookies

In [ ]:
def get_boardgame_list(cookies:dict):
    bg_list_pg_url = "https://boardgamegeek.com/data_dumps/bg_ranks"
    resp = requests.get(bg_list_pg_url, cookies=cookies)
    soup = BeautifulSoup(resp.content, "html.parser")
    bg_list_url = soup.find("div", {"id": "maincontent"})("a")[0]["href"]
    bg_list_zip = requests.get(bg_list_url)
    with ZipFile(BytesIO(bg_list_zip.content)) as archive:
        with archive.open("boardgames_ranks.csv") as csv:
            df_bg_list = pd.read_csv(csv)
            df_bg_list["queried_at_utc"] = datetime.now().replace(microsecond=0).isoformat()
            return df_bg_list

In [ ]:
def get_boardgame_raw_data(boardgame_df: pd.DataFrame):
    boardgame_ids = boardgame_df["id"].tolist()
    boardgame_list = []
    batch_size = 3
    for batch_num in range(math.ceil(len(boardgame_ids) / batch_size)):
        batch_ids = boardgame_ids[batch_num * batch_size : (batch_num + 1) * batch_size]
        batch_ids = [str(x) for x in batch_ids]
        bg_info_url = f"https://www.boardgamegeek.com/xmlapi2/thing?type=boardgame&stats=1&versions=1&ratingcomments=1&pagesize=100&page=1&id={','.join(batch_ids)}"
        bgg_response = requests.get(bg_info_url)
        soup_xml = BeautifulSoup(bgg_response.content, "xml")
        games_xml_list = soup_xml.find_all("item", attrs={"type": "boardgame"})
        ratings_count_dict = {}
        for game_xml in games_xml_list:
            game_dict = extract_basic_game_info(game_xml=game_xml)
            game_dict = extract_polls(game_dict=game_dict, game_xml=game_xml)
            game_dict = extract_poll_player_count(
                game_dict=game_dict, game_xml=game_xml
            )
            game_dict = extract_version_info(game_dict=game_dict, game_xml=game_xml)
            if game_dict["numratings"] > 0:
                game_dict = extract_ratings(game_dict=game_dict, game_xml=game_xml)
            ratings_count_dict[game_dict["game_id"]] = game_dict["numratings"]
            max_ratings_page = math.ceil(max(ratings_count_dict.values()) / 100)
            boardgame_list.append(game_dict)
        for page_num in range(2, max_ratings_page):
            # Only grab the pages for games which have enough ratings to be on the page num
            batch_ids_ratings = [
                str(x)
                for x in ratings_count_dict.keys()
                if math.ceil(ratings_count_dict[x] / 100) >= page_num
            ]
            bg_rating_url = f"https://www.boardgamegeek.com/xmlapi2/thing?type=boardgame&ratingcomments=1&pagesize=100&page={page_num}&id={','.join(batch_ids_ratings)}"
            bgg_rating_response = requests.get(bg_rating_url)
            soup_rating_xml = BeautifulSoup(bgg_rating_response.content, "xml")
            ratings_xml_list = soup_rating_xml.find_all(
                "item", attrs={"type": "boardgame"}
            )
            for game_xml in ratings_xml_list:
                game_dict = extract_ratings(game_dict=game_dict, game_xml=game_xml)
        break
    return boardgame_list

In [ ]:
def extract_basic_game_info(game_xml: bs4.element.Tag):
    game_dict = {
        "game_id": game_xml["id"],
    }
    if game_xml.find("image") is not None:
        game_dict["thumbnail"] = game_xml.find("thumbnail").text
        game_dict["image"] = game_xml.find("image").text
    values_int = [
        "minplayers",
        "maxplayers",
        "playingtime",
        "minplaytime",
        "maxplaytime",
        "minage",
    ]
    for vals in values_int:
        if game_xml.find(vals) is not None:
            game_dict[vals] = game_xml.find(vals)["value"]
    link_categ = [
        "boardgamecategory",
        "boardgamemechanic",
        "boardgamefamily",
        "boardgameexpansion",
        "boardgameartist",
        "boardgamecompilation",
        "boardgameimplementation",
        "boardgamedesigner",
        "boardgamepublisher",
        "boardgameintegration",
    ]
    for categ in link_categ:
        game_dict[categ] = [
            x["value"] for x in game_xml.find_all("link", {"type": categ})
        ]
    stats_float = ["stddev", "median", "averageweight"]
    for stat in stats_float:
        if game_xml.find(stat) is not None:
            game_dict[stat] = float(game_xml.find(stat)["value"])
    stats_int = [
        "owned",
        "trading",
        "wanting",
        "wishing",
        "numcomments",
        "numweights",
    ]
    for stat in stats_int:
        if game_xml.find(stat) is not None:
            game_dict[stat] = int(game_xml.find(stat)["value"])
    if game_xml.find("comments") is not None:
        game_dict["ratings"] = int(game_xml.find("comments")["totalitems"])
    else:
        game_dict["ratings"] = 0
    return game_dict

In [ ]:
def extract_polls(game_dict:dict, game_xml: bs4.element.Tag):
    for poll_name in ["suggested_playerage", "language_dependence"]:
        if poll_name == "suggested_playerage":
            raw_value_col = "value"
        elif poll_name == "language_dependence":
            raw_value_col = "level"
        poll = game_xml.find("poll", attrs={"name": poll_name})
        vote_count = int(poll.attrs["totalvotes"])
        if vote_count > 0:
            result_dict = {"total_votes": vote_count}
            cum_votes = 0
            suggested_prcnt = {}
            suggested_prcnt_col = "value"
            for result_val in poll.find("results").find_all("result"):
                num_votes = int(result_val["numvotes"]) 
                cum_votes += num_votes
                cum_prcnt = round(cum_votes/vote_count*100)
                result_dict[result_val[raw_value_col]] = num_votes
                sugg_prcnt_val = result_val[suggested_prcnt_col]
                if cum_prcnt >= 75:
                    if "75 percent" not in suggested_prcnt:
                        suggested_prcnt["75 percent"] = sugg_prcnt_val
                    if "50 percent" not in suggested_prcnt:
                        suggested_prcnt["50 percent"] = sugg_prcnt_val
                    if "25 percent" not in suggested_prcnt:
                        suggested_prcnt["25 percent"] = sugg_prcnt_val
                elif cum_prcnt >= 50:
                    suggested_prcnt["50 percent"] = sugg_prcnt_val
                    if "25 percent" not in suggested_prcnt:
                        suggested_prcnt["25 percent"] = sugg_prcnt_val
                elif cum_prcnt >= 25:
                    suggested_prcnt["25 percent"] = sugg_prcnt_val
        else:
            result_dict = {}
            suggested_prcnt = {}
        game_dict[poll_name] = result_dict
        game_dict[f"{poll_name}_quartiles"] = suggested_prcnt  
    return game_dict

In [ ]:
def extract_poll_player_count(game_dict:dict, game_xml: bs4.element.Tag):
    player_count_poll = game_xml.find("poll", attrs={"name": "suggested_numplayers"})
    result_dict = {"total_votes": int(player_count_poll.attrs["totalvotes"])}
    player_count_results = player_count_poll.find_all("results")
    game_dict["player_count_recs"] = {}
    for player_count in player_count_results:
        num_players = player_count.attrs["numplayers"]
        player_count_values = {
            x.attrs["value"]: int(x.attrs["numvotes"])
            for x in player_count.find_all("result")
        }
        play_count_rec = max(player_count_values, key=player_count_values.get)
        if play_count_rec in game_dict["player_count_recs"]:
            game_dict["player_count_recs"][play_count_rec].append(num_players)
        else:
            game_dict["player_count_recs"][play_count_rec] = [num_players]
        result_dict[num_players] = player_count_values
        result_dict[num_players]["total_votes"] = sum(
            int(x.attrs["numvotes"]) for x in player_count.find_all("result")
        )
    game_dict["suggested_numplayers"] = result_dict  
    return game_dict

In [ ]:
def extract_version_info(game_dict: dict, game_xml: bs4.element.Tag):
    version_items = game_xml.find_all("item", attrs={"type": "boardgameversion"})
    version_list = []
    for vrs in version_items:
        version_dict = {
            "version_id": int(vrs["id"]),
            "version_nickname": vrs.find("name", attrs={"type": "primary"})[
                "value"
            ],
            "width": round(float(vrs.find("width")["value"])),
            "length": round(float(vrs.find("length")["value"])),
            "depth": round(float(vrs.find("depth")["value"])),
            "year_published": round(float(vrs.find("yearpublished")["value"])),
            "language": vrs.find("link", attrs={"type": "language"})["value"].lower(),
        }
        if vrs.find("thumbnail") is not None:
            version_dict["thumbnail"] = vrs.find("thumbnail").text
            version_dict["image"] = vrs.find("image").text
        if version_dict["width"] > 0:
            version_list.append(version_dict)
    if len(version_list) > 0:
        game_dict["versions"] = version_list
    return game_dict

In [ ]:
def extract_ratings(game_dict: dict, game_xml: bs4.element.Tag):
    ratings_list = game_xml.find_all("comment")
    if "ratings" not in game_dict:
        game_dict["ratings"] = {}
    for rating in ratings_list:
        # round the rating to the nearest 0.5
        rating_round = round(2*float(rating["rating"]))/2
        if rating_round not in game_dict["ratings"]:
            game_dict["ratings"][rating_round] =  [rating["username"]]
        else:
            game_dict["ratings"][rating_round].append(rating["username"])
    return game_dict

In [ ]:
def get_boardgame_raw_data(boardgame_df: pd.DataFrame):
    boardgame_ids = boardgame_df["id"].tolist()
    boardgame_master_dict = {}
    batch_size = 3
    for batch_num in range(math.ceil(len(boardgame_ids) / batch_size)):
        batch_ids = boardgame_ids[batch_num * batch_size : (batch_num + 1) * batch_size]
        batch_ids = [str(x) for x in batch_ids]
        bg_info_url = f"https://www.boardgamegeek.com/xmlapi2/thing?type=boardgame&stats=1&versions=1&ratingcomments=1&pagesize=100&page=1&id={','.join(batch_ids)}"
        bgg_response = requests.get(bg_info_url)
        soup_xml = BeautifulSoup(bgg_response.content, "xml")
        games_xml_list = soup_xml.find_all("item", attrs={"type": "boardgame"})
        ratings_count_dict = {}
        for game_xml in games_xml_list:
            game_dict = extract_basic_game_info(game_xml=game_xml)
            game_dict = extract_polls(game_dict=game_dict, game_xml=game_xml)
            game_dict = extract_poll_player_count(
                game_dict=game_dict, game_xml=game_xml
            )
            game_dict = extract_version_info(game_dict=game_dict, game_xml=game_xml)
            if game_dict["numratings"] > 0:
                game_dict = extract_ratings(game_dict=game_dict, game_xml=game_xml)
            ratings_count_dict[game_dict["game_id"]] = game_dict["numratings"]
            max_ratings_page = math.ceil(max(ratings_count_dict.values()) / 100)
            boardgame_master_dict[game_dict["game_id"]] = game_dict
        for page_num in range(2, max_ratings_page + 1):
            print(page_num)
            # Only grab the pages for games which have enough ratings to be on the page num
            batch_ids_ratings = [
                str(x)
                for x in ratings_count_dict.keys()
                if math.ceil(ratings_count_dict[x] / 100) >= page_num
            ]
            bg_rating_url = f"https://www.boardgamegeek.com/xmlapi2/thing?type=boardgame&ratingcomments=1&pagesize=100&page={page_num}&id={','.join(batch_ids_ratings)}"
            bgg_rating_response = requests.get(bg_rating_url)
            soup_rating_xml = BeautifulSoup(bgg_rating_response.content, "xml")
            ratings_xml_list = soup_rating_xml.find_all(
                "item", attrs={"type": "boardgame"}
            )
            for game_xml in ratings_xml_list:
                game_dict = boardgame_master_dict[game_xml["id"]]
                boardgame_master_dict[game_dict["game_id"]] = extract_ratings(
                    game_dict=game_dict, game_xml=game_xml
                )
        break
    return list(boardgame_master_dict.values())


def extract_basic_game_info(game_xml: bs4.element.Tag):
    game_dict = {
        "game_id": game_xml["id"],
    }
    if game_xml.find("image") is not None:
        game_dict["thumbnail"] = game_xml.find("thumbnail").text
        game_dict["image"] = game_xml.find("image").text
    values_int = [
        "minplayers",
        "maxplayers",
        "playingtime",
        "minplaytime",
        "maxplaytime",
        "minage",
    ]
    for vals in values_int:
        if game_xml.find(vals) is not None:
            game_dict[vals] = game_xml.find(vals)["value"]
    link_categ = [
        "boardgamecategory",
        "boardgamemechanic",
        "boardgamefamily",
        "boardgameexpansion",
        "boardgameartist",
        "boardgamecompilation",
        "boardgameimplementation",
        "boardgamedesigner",
        "boardgamepublisher",
        "boardgameintegration",
    ]
    for categ in link_categ:
        game_dict[categ] = [
            x["value"] for x in game_xml.find_all("link", {"type": categ})
        ]
    stats_float = ["stddev", "median", "averageweight"]
    for stat in stats_float:
        if game_xml.find(stat) is not None:
            game_dict[stat] = float(game_xml.find(stat)["value"])
    stats_int = [
        "owned",
        "trading",
        "wanting",
        "wishing",
        "numcomments",
        "numweights",
    ]
    for stat in stats_int:
        if game_xml.find(stat) is not None:
            game_dict[stat] = int(game_xml.find(stat)["value"])
    if game_xml.find("comments") is not None:
        game_dict["numratings"] = int(game_xml.find("comments")["totalitems"])
    else:
        game_dict["numratings"] = 0
    return game_dict


def extract_polls(game_dict: dict, game_xml: bs4.element.Tag):
    for poll_name in ["suggested_playerage", "language_dependence"]:
        if poll_name == "suggested_playerage":
            raw_value_col = "value"
        elif poll_name == "language_dependence":
            raw_value_col = "level"
        poll = game_xml.find("poll", attrs={"name": poll_name})
        vote_count = int(poll.attrs["totalvotes"])
        if vote_count > 0:
            result_dict = {"total_votes": vote_count}
            cum_votes = 0
            suggested_prcnt = {}
            suggested_prcnt_col = "value"
            for result_val in poll.find("results").find_all("result"):
                num_votes = int(result_val["numvotes"])
                cum_votes += num_votes
                cum_prcnt = round(cum_votes / vote_count * 100)
                result_dict[result_val[raw_value_col]] = num_votes
                sugg_prcnt_val = result_val[suggested_prcnt_col]
                if cum_prcnt >= 75:
                    if "75 percent" not in suggested_prcnt:
                        suggested_prcnt["75 percent"] = sugg_prcnt_val
                    if "50 percent" not in suggested_prcnt:
                        suggested_prcnt["50 percent"] = sugg_prcnt_val
                    if "25 percent" not in suggested_prcnt:
                        suggested_prcnt["25 percent"] = sugg_prcnt_val
                elif cum_prcnt >= 50:
                    suggested_prcnt["50 percent"] = sugg_prcnt_val
                    if "25 percent" not in suggested_prcnt:
                        suggested_prcnt["25 percent"] = sugg_prcnt_val
                elif cum_prcnt >= 25:
                    suggested_prcnt["25 percent"] = sugg_prcnt_val
        else:
            result_dict = {}
            suggested_prcnt = {}
        game_dict[poll_name] = result_dict
        game_dict[f"{poll_name}_quartiles"] = suggested_prcnt
    return game_dict


def extract_poll_player_count(game_dict: dict, game_xml: bs4.element.Tag):
    player_count_poll = game_xml.find("poll", attrs={"name": "suggested_numplayers"})
    result_dict = {"total_votes": int(player_count_poll.attrs["totalvotes"])}
    player_count_results = player_count_poll.find_all("results")
    game_dict["player_count_recs"] = {}
    for player_count in player_count_results:
        num_players = player_count.attrs["numplayers"]
        player_count_values = {
            x.attrs["value"]: int(x.attrs["numvotes"])
            for x in player_count.find_all("result")
        }
        play_count_rec = max(player_count_values, key=player_count_values.get)
        if play_count_rec in game_dict["player_count_recs"]:
            game_dict["player_count_recs"][play_count_rec].append(num_players)
        else:
            game_dict["player_count_recs"][play_count_rec] = [num_players]
        result_dict[num_players] = player_count_values
        result_dict[num_players]["total_votes"] = sum(
            int(x.attrs["numvotes"]) for x in player_count.find_all("result")
        )
    game_dict["suggested_numplayers"] = result_dict
    return game_dict


def extract_version_info(game_dict: dict, game_xml: bs4.element.Tag):
    version_items = game_xml.find_all("item", attrs={"type": "boardgameversion"})
    version_list = []
    for vrs in version_items:
        version_dict = {
            "version_id": int(vrs["id"]),
            "version_nickname": vrs.find("name", attrs={"type": "primary"})["value"],
            "width": round(float(vrs.find("width")["value"])),
            "length": round(float(vrs.find("length")["value"])),
            "depth": round(float(vrs.find("depth")["value"])),
            "year_published": round(float(vrs.find("yearpublished")["value"])),
            "language": vrs.find("link", attrs={"type": "language"})["value"].lower(),
        }
        if vrs.find("thumbnail") is not None:
            version_dict["thumbnail"] = vrs.find("thumbnail").text
            version_dict["image"] = vrs.find("image").text
        if version_dict["width"] > 0:
            version_list.append(version_dict)
    if len(version_list) > 0:
        game_dict["versions"] = version_list
    return game_dict


def extract_ratings(game_dict: dict, game_xml: bs4.element.Tag):
    ratings_list = game_xml.find_all("comment")
    if "ratings" not in game_dict:
        game_dict["ratings"] = {}
    for rating in ratings_list:
        # round the rating to the nearest 0.5
        rating_round = round(2 * float(rating["rating"])) / 2
        if rating_round not in game_dict["ratings"]:
            game_dict["ratings"][rating_round] = [rating["username"]]
        else:
            game_dict["ratings"][rating_round].append(rating["username"])
    return game_dict


In [ ]:
df_bg_list.loc[df_bg_list["id"].isin([2766, 2767, 2768])]

In [ ]:
2767

In [ ]:
df_bg_list.loc[df_bg_list["usersrated"]<500]

In [ ]:
bg_raw_data = get_boardgame_raw_data(df_bg_list.loc[df_bg_list["id"].isin([2766, 2767, 2768])])

In [ ]:
df_bg_raw_data = pd.DataFrame(bg_raw_data)
df_bg_raw_data.head()

In [ ]:
sum([len(v) for k,v in df_bg_raw_data["ratings"].iloc[2].items()])

# Code

## Pull ranks

In [ ]:
df_tmp = pd.read_csv("../../data/processed/processed_games_language_dependence_1748995280.csv", sep="|", escapechar="\\")

In [ ]:
df_tmp.dtypes

In [ ]:
cookies = crawler.get_driver_and_cookies()

In [ ]:
df_boardgame_ranks = crawler.get_boardgame_ranks(cookies=cookies, save_file=True)

In [ ]:
df_ranks = pd.read_csv("../../data/crawler/boardgame_ranks_20250526.csv")
df_ranks.head()

## pulling basic game info

In [ ]:
# df_tmp = pd.read_csv(
#     "../../data/boardgame_data_raw_1748297111.csv",
#     sep="|",
#     escapechar="\\"
# )
# for col in [
#     "suggested_playerage",
#     "suggested_playerage_quartiles",
#     "language_dependence",
#     "language_dependence_quartiles",
#     "player_count_recs",
#     "suggested_numplayers",
#     "versions",
#     "ratings",
# ]:
#     df_tmp[col] = df_tmp[col].apply(lambda x: literal_eval(x))
# df_tmp.to_parquet("../../data/boardgame_data_raw_1748276896.parquet")    

In [ ]:
df_ranks.columns

In [ ]:
(df_parq["description"].iloc[0])

In [ ]:
df_parq = pd.read_parquet("../../data/crawler/boardgame_data_1749186734.parquet")
print(df_parq.shape)
df_parq.head()

In [ ]:
df_parq["description"].iloc[0]

In [ ]:
df_parq = pd.read_parquet("../../data/crawler/boardgame_simple_data_1749187980.parquet")
print(df_parq.shape)
df_parq.head()

In [ ]:
combined_dict = {}
for d in df_parq["boardgamecategory"]:
    combined_dict.update(d)

In [ ]:
boardgamecategory_rows = []
col = "boardgamecategory"
for _, row in df_parq.iterrows():
    if isinstance(row[col], dict):  # Ensure it's a dict
        for k, v in row[col].items():
            boardgamecategory_rows.append({"game_id": row["id"], f"{col}_id": k, f"{col}_name": v})
df_category_row = pd.DataFrame(boardgamecategory_rows)

In [ ]:
df_merged.head()

In [ ]:
df_full = pd.read_parquet("../../data/crawler/boardgame_data_1748875957.parquet")

In [ ]:
poss_cat = [
    "boardgamecategory",
    "boardgamemechanic",
    "boardgamefamily",
    "boardgameexpansion",
    "boardgameartist",
    "boardgamecompilation",
    "boardgameimplementation",
    "boardgamedesigner",
    "boardgamepublisher",
    "boardgameintegration",
]
col = "boardgameexpansion"
df_exploded = pd.DataFrame(
    [
        {"game_id": row_id, f"{col}_id": k, f"{col}_name": v}
        for row_id, d in zip(df_parq["id"], df_parq[col])
        if isinstance(d, dict)  # ensure it's a dict
        for k, v in d.items()
    ]
)
df_exploded[f"{col}_id"] = df_exploded[f"{col}_id"].astype(int)
df_exploded.head()

In [ ]:
df_parq["language_dependence"].iloc[3]

In [ ]:
df_lang = df_full.loc[~(df_full["language_dependence"]=={}), ["id", "language_dependence"]]
df_lang["game_total_votes"] = df_lang["language_dependence"].apply(lambda x: x["total_votes"])
df_lang = df_lang.loc[df_lang["game_total_votes"]>10]
df_lang

In [ ]:
df_lang = df_full.loc[~(df_full["language_dependence"]=={}), ["id", "language_dependence"]]
df_lang["game_total_votes"] = df_lang["language_dependence"].apply(lambda x: x["total_votes"])
df_lang = df_lang.loc[df_lang["game_total_votes"]>10]
col = "language_dependence"
df_exploded = pd.DataFrame(
    [
        {"id": row_id, col: k, "votes": v}
        for row_id, d in zip(df_lang["id"], df_lang[col])
        if isinstance(d, dict)  # ensure it's a dict
        for k, v in d.items()
    ]
)
df_exploded = df_exploded.loc[(df_exploded[col] != "total_votes")]
df_exploded["language_dependence"] = df_exploded["language_dependence"].astype(int)
df_min = (
    df_exploded[["id", "language_dependence"]]
    .groupby("id")
    .min()
    .reset_index()
    .rename(columns={"language_dependence": "lang_min"})
)
df_exploded = df_exploded.merge(df_min, how="left", on="id")
df_exploded["language_dependence_norm"] = (
    df_exploded["language_dependence"] - df_exploded["lang_min"] + 1
)
df_exploded = df_exploded.pivot(index="id", columns="language_dependence_norm", values="votes")
df_exploded["total_votes"] = df_exploded.sum(axis=1)
df_exploded["language_dependency"] = df_exploded[[1,2,3,4,5]].idxmax(axis=1)
df_exploded = df_exploded.reset_index()
df_exploded.columns.name=None
# df_exploded = pd.concat(
#     [df_lang.drop(columns="votes"), pd.json_normalize(df_exploded["votes"])], axis=1
# )
# df_exploded = df_exploded.loc[
#     (df_exploded[col] != "total_votes") & ~(df_exploded[col].str.contains("+", regex=False))
# ]
# df_exploded = df_exploded.merge(df_sugg_players[["id", "game_total_votes"]], on="id", how="left")
# df_exploded.columns = [x.lower().replace(" ", "_") for x in df_exploded.columns]
# for col in df_exploded.columns:
#     df_exploded[col] = df_exploded[col].astype(pd.Int64Dtype())
# df_exploded["recommendation"] = df_exploded[["best", "recommended", "not_recommended"]].idxmax(
#     axis=1
# )
# df_exploded = df_exploded.rename(columns={"suggested_numplayers": "players", "id": "game_id"})
df_exploded.head(20)

In [ ]:
df_exploded.columns

In [ ]:
df_exploded["language_dependence_norm"].value_counts()

In [ ]:
col = "language_dependence"
# df_exploded = pd.DataFrame(
#     [
#         {"id": row_id, col: k, "votes": v}
#         for row_id, d in zip(df_lang["id"], df_lang[col])
#         if isinstance(d, dict)  # ensure it's a dict
#         for k, v in d.items()
#     ]
# )
df_exploded = pd.concat(
    [df_lang.drop(columns=col), pd.json_normalize(df_lang[col])], axis=1
)
# df_exploded = df_exploded.loc[
#     (df_exploded[col] != "total_votes") & ~(df_exploded[col].str.contains("+", regex=False))
# ]
# df_exploded = df_exploded.merge(df_sugg_players[["id", "game_total_votes"]], on="id", how="left")
# df_exploded.columns = [x.lower().replace(" ", "_") for x in df_exploded.columns]
# for col in df_exploded.columns:
#     df_exploded[col] = df_exploded[col].astype(pd.Int64Dtype())
# df_exploded["recommendation"] = df_exploded[["best", "recommended", "not_recommended"]].idxmax(
#     axis=1
# )
# df_exploded = df_exploded.rename(columns={"suggested_numplayers": "players", "id": "game_id"})
df_exploded.head()

In [ ]:
col = "versions"
df_exploded = pd.DataFrame(
    [
        {"id": row_id, col: k, "votes": v}
        for row_id, d in zip(df_sugg_players["id"], df_sugg_players[col])
        if isinstance(d, dict)  # ensure it's a dict
        for k, v in d.items()
    ]
)
df_exploded = pd.concat(
    [df_exploded.drop(columns="votes"), pd.json_normalize(df_exploded["votes"])], axis=1
)

In [ ]:
df_versions = df_full.loc[df_full["versions"].notna(),["id","versions"]].explode("versions").reset_index(drop=True)
df_versions = pd.concat(
    [df_versions.drop(columns="versions"), pd.json_normalize(df_versions["versions"])], axis=1
)
df_versions = df_versions.rename(columns={"id":"game_id"})
df_versions.head()

In [ ]:
df_versions.dtypes

In [ ]:
pd.json_normalize(df_versions["versions"])

In [ ]:
df_sugg_players["suggested_numplayers"].iloc[-1]

In [ ]:
df_sugg_players.loc[df_sugg_players["total_votes"]>10].shape

In [ ]:
df_full.loc[~(df_full["suggested_numplayers"]=={})]

In [ ]:
df_full["suggested_numplayers"].apply(lambda x: x.get("total_votes"))

In [ ]:
df_exploded.dtypes

In [ ]:
df_merged["player_count_recs"].iloc[4]

In [ ]:
df_merged["suggested_numplayers"].iloc[4]

In [ ]:
del(df_category_row)

In [ ]:
df_tmp = df_parq[["id", "boardgamecategory"]].copy()
# Step 1: Convert dictionary into Series (key-value pairs as rows)
df_exploded = df_tmp.explode(df_tmp["boardgamecategory"].apply(lambda d: list(d.items())))

# Step 2: Extract keys and values from the tuples
df_exploded[["key", "value"]] = pd.DataFrame(
    df_exploded["boardgamecategory"].tolist(), index=df_exploded.index
)

# df_tmp["boardgamecategory"] = df_tmp["boardgamecategory"].apply(lambda x: list(x.keys()))
# # Step 1: Explode the list column
# exploded = df_tmp.explode('boardgamecategory')

# # Step 2: Create one-hot encoding
# one_hot = pd.get_dummies(exploded['boardgamecategory'])

# # Step 3: Group back to original rows (sum by original index)
# one_hot_encoded = one_hot.groupby(exploded.index).sum()

# # Step 4: Combine with original DataFrame
# result = df_tmp.drop(columns='boardgamecategory').join(one_hot_encoded)

In [ ]:
df_tmp

In [ ]:
df_tmp.to_dict(orient="records")

In [ ]:
len(combined_dict.keys())

In [ ]:
pd.Series(df_tmp['numratings'].values, index=df_tmp['game_id']).to_dict()

In [ ]:
int(df_tmp["ratings_pulled"].min()/100)

In [ ]:
import importlib
importlib.reload(crawler)

In [ ]:
import requests
from bs4 import BeautifulSoup
batch_ids = ['245952']
bg_info_url = f"https://www.boardgamegeek.com/xmlapi2/thing?type=boardgame,boardgameexpansion&stats=1&versions=1&ratingcomments=1&pagesize=100&page=1&id={','.join(batch_ids)}"
print(bg_info_url)
# bgg_response = requests.get(bg_info_url)
# soup_xml = BeautifulSoup(bgg_response.content, "xml")
# games_xml_list = soup_xml.find_all("item", attrs={"type": ["boardgame", "boardgameexpansion"]})
# len(games_xml_list)

In [ ]:
# batch_ids = [str(x) for x in df_ranks.iloc[105293:105313]["id"].tolist()]
batch_ids = '13'
bg_info_url = f"https://www.boardgamegeek.com/xmlapi2/thing?type=boardgame,boardgameexpansion&stats=1&versions=1&ratingcomments=1&pagesize=100&page=1&id={','.join(batch_ids)}"
bgg_response = requests.get(bg_info_url)
soup_xml = BeautifulSoup(bgg_response.content, "xml")
games_xml_list = soup_xml.find_all("item", attrs={"type": ["boardgame", "boardgameexpansion"]})

In [ ]:
import html
print(html.unescape(games_xml_list[0].find("description").text))

In [ ]:
games_xml_list

## Merge data

In [ ]:
df_merged = df_ranks.head().drop(columns=['queried_at_utc']).merge(df_parq.head(), on="id", how="inner")
games_data = df_merged.to_dict(orient="records")

In [ ]:
df_parq["minplayers"].value_counts(dropna=False)

In [ ]:
import numpy as np

In [ ]:
df_test = pd.DataFrame({"col": [1.23, 2, 3, np.nan, None]})
df_test["col"] = df_test["col"].fillna(value=pd.NA).astype(pd.Float64Dtype())
df_test

In [ ]:
df_test.to_dict(orient="records")

In [ ]:
df_parq = pd.read_parquet("../../data/crawler/boardgame_data_1748875957.parquet")
df_parq.shape

In [ ]:
df_parq_simple = pd.read_parquet("../../data/crawler/boardgame_simple_data_1749221749.parquet", columns=["id", "description"])
df_parq_simple.shape

In [ ]:
df_parq_merged = df_parq.merge(df_parq_simple, on="id", how="left")
df_parq_merged.shape

In [ ]:
from time import sleep, time
int(time())

In [ ]:
df_parq_merged.columns

In [ ]:
[
    "id",
    "thumbnail",
    "image",
    "minplayers",
    "maxplayers",
    "playingtime",
    "minplaytime",
    "maxplaytime",
    "minage",
    "boardgamecategory",
    "boardgamemechanic",
    "boardgamefamily",
    "boardgameexpansion",
    "boardgameartist",
    "boardgamecompilation",
    "boardgameimplementation",
    "boardgamedesigner",
    "boardgamepublisher",
    "boardgameintegration",
    "stddev",
    "median",
    "averageweight",
    "owned",
    "trading",
    "wanting",
    "wishing",
    "numcomments",
    "numweights",
    "numratings",
    "suggested_playerage",
    "suggested_playerage_quartiles",
    "language_dependence",
    "language_dependence_quartiles",
    "player_count_recs",
    "suggested_numplayers",
    "versions",
    "description",
]

In [ ]:
query_time = int(time())
df_parq_merged = df_parq_merged[
    [
        "id",
        "thumbnail",
        "image",
        "description",
        "minplayers",
        "maxplayers",
        "playingtime",
        "minplaytime",
        "maxplaytime",
        "minage",
        "boardgamecategory",
        "boardgamemechanic",
        "boardgamefamily",
        "boardgameexpansion",
        "boardgameartist",
        "boardgamecompilation",
        "boardgameimplementation",
        "boardgamedesigner",
        "boardgamepublisher",
        "boardgameintegration",
        "stddev",
        "median",
        "averageweight",
        "owned",
        "trading",
        "wanting",
        "wishing",
        "numcomments",
        "numweights",
        "numratings",
        "suggested_playerage",
        "suggested_playerage_quartiles",
        "language_dependence",
        "language_dependence_quartiles",
        "player_count_recs",
        "suggested_numplayers",
        "versions",        
    ]
]
df_parq_merged.to_parquet(f"../../data/crawler/boardgame_data_{query_time}.parquet")

In [ ]:
df_parq_merged.loc[df_parq_merged["description"].isna()]

## Get ratings data

In [ ]:
import importlib
importlib.reload(crawler)

In [ ]:
df_bg_data = pd.read_parquet("../../data/crawler/boardgame_data_1748311285.parquet")
print(df_bg_data.shape)
df_bg_data.head()

In [ ]:
df_ratings = pd.read_parquet("../../data/crawler/boardgame_ratings_1748644891.parquet")
print(df_ratings.shape)
# df_ratings.head()
# 1420

In [ ]:
df_bg_data.loc[df_bg_data["game_id"]=='360899', "numratings"]

In [ ]:
df_tmp2 = df_ratings.loc[df_ratings["game_id"]=="360899"].drop(columns=["game_id"]).fillna("")
for col in df_tmp2.columns:
    df_tmp2[col] = df_tmp2[col].apply(len)
df_tmp2.sum(axis=1)

In [ ]:
# df_bg_data.loc[(df_bg_data["numratings"]>=520) & (df_bg_data["numratings"]<600)].shape #563
# df_bg_data.loc[(df_bg_data["numratings"]>=600) & (df_bg_data["numratings"]<700)].shape #572
# df_bg_data.loc[(df_bg_data["numratings"]>=700) & (df_bg_data["numratings"]<800)].shape #451
# df_bg_data.loc[(df_bg_data["numratings"]>=800) & (df_bg_data["numratings"]<900)].shape #336
# df_bg_data.loc[(df_bg_data["numratings"]>=900) & (df_bg_data["numratings"]<1000)].shape #323
# df_bg_data.loc[(df_bg_data["numratings"]>=2000) & (df_bg_data["numratings"]<8000)].shape #451
df_bg_data.loc[(df_bg_data["numratings"]>=8000)].shape #451

In [ ]:
df_ratings.shape

In [ ]:
df_ratings.head()

### Convert the ratings dataframe into the proper format

In [ ]:
df_ratings.shape

In [ ]:
rows = []
for _, row in df_ratings.head().iterrows():
    for rating in df_ratings.columns:
        users = row[rating]
        game_id = row["game_id"]
        if isinstance(users, list):
            for user in users:
                rows.append((user, game_id, float(rating)))
# Convert to DataFrame
ratings_df = pd.DataFrame(rows, columns=['user', 'game', 'rating'])   


In [ ]:
# Step 1: Flatten and explode data
rows = []
for _, row in df_ratings.iterrows():
    for rating in df_ratings.columns:
        users = row[rating]
        game_id = row["game_id"]
        if isinstance(users, list):
            for user in users:
                rows.append((user, game_id, float(rating)))
# Convert to DataFrame
ratings_df = pd.DataFrame(rows, columns=['user', 'game', 'rating'])   

# Count ratings per user
user_counts = ratings_df['user'].value_counts()

# Filter users with at least 3 ratings
valid_users = user_counts[user_counts >= 3].index

# Filter the dataframe
filtered_ratings_df = ratings_df[ratings_df['user'].isin(valid_users)].copy()

# Step 2: Encode users and games as integer IDs
user_encoder = LabelEncoder()
game_encoder = LabelEncoder()

filtered_ratings_df['user_id'] = user_encoder.fit_transform(filtered_ratings_df['user'])
filtered_ratings_df['game_id'] = game_encoder.fit_transform(filtered_ratings_df['game'])

# Step 3: Create sparse matrix
user_ids = filtered_ratings_df['user_id'].values
game_ids = filtered_ratings_df['game_id'].values
ratings = filtered_ratings_df['rating'].values

ratings_sparse_matrix = coo_matrix((ratings, (user_ids, game_ids)))
print(ratings_sparse_matrix.shape)
# del(filtered_ratings_df)
# del(ratings_df)
# del(rows)

In [ ]:
sparse_matrix_csr = ratings_sparse_matrix.tocsr()

In [ ]:
item_user_matrix = csr_matrix(ratings_sparse_matrix.T)

model = AlternatingLeastSquares(factors=50, regularization=0.1, iterations=20)
model.fit(item_user_matrix)

In [ ]:
[i for i in similar]

In [ ]:
filtered_ratings_df

In [ ]:
# Pick a game_id (integer, not original string)
game_id = 1713  # example

# Get similar items (game IDs and similarity scores)
similar = model.similar_items(game_id, N=5)

# Decode back to original game names
# similar_games = [(item_encoder.inverse_transform([i])[0], score) for i, score in similar]
similar_games = list(zip(similar[0],similar[1]))
for game, score in similar_games:
    print(f"{game}: similarity {score:.4f}")

In [ ]:
model.similar_items(itemid)

In [ ]:
filtered_ratings_df["game"]

In [ ]:
filtered_ratings_df.loc[filtered_ratings_df["user_id"]==140033]

In [ ]:
filtered_ratings_df.loc[filtered_ratings_df["game"]=='140033']

In [ ]:
ratings_sparse_matrix.

In [ ]:
# Melt the dataframe to long format
long_format = df_ratings.head(2000).melt(id_vars='game_id', var_name='rating', value_name='users')

# Drop rows where users is empty
long_format = long_format.dropna(subset=['users'])

# Explode the user lists
long_format = long_format.explode('users')

# Convert score to int
long_format['rating'] = long_format['rating'].astype(float)

# Now pivot to user-item matrix
user_item_matrix = long_format.pivot_table(index='users', columns='game_id', values='rating')

del long

In [ ]:
user_item_matrix

In [ ]:
df_bg_data_raw

In [ ]:
pd.DataFrame({"game_id":df_ranks_out["game_id"].tolist(), "ratings_pulled":df_ranks_len.sum(axis=1).tolist()})

In [ ]:
df_bg_list.loc[df_bg_list["id"].isin([int(x) for x in batch_ids]), "usersrated"]

In [ ]:
id_list = [int(x) for x in batch_ids[:1]]
math.ceil(df_bg_list.loc[df_bg_list["id"].isin([int(x) for x in batch_ids]), "usersrated"].max()/10)

In [ ]:
batch_ids

In [ ]:
cookies = get_driver_and_cookies()
df_bg_list = get_bg_list(cookies=cookies)

In [ ]:
bg_list = get_boardgame_data(boardgame_ids=df_bg_list["id"])

In [ ]:
bg_list

In [ ]:
5

In [ ]:
df = pd.DataFrame(bg_list)
df.head()

In [ ]:
df["ratings"].iloc[0]

In [ ]:
soup_xml = get_boardgame_info(boardgame_ids=df_bg_list["id"].astype(str).tolist())

In [ ]:
batch_ids

In [ ]:
# batch_ids = boardgame_ids[batch_num * batch_size : (batch_num + 1) * batch_size]
# batch_ids = [str(x) for x in batch_ids]
batch_ids = ['224517', '161936', '342942', '174430', '233078', '447122']
bg_info_url = f"https://www.boardgamegeek.com/xmlapi2/thing?type=boardgame&stats=1&versions=1&ratingcomments=1&pagesize=10&page=1&id={','.join(batch_ids)}"
bgg_response = requests.get(bg_info_url)
soup_xml = BeautifulSoup(bgg_response.content, "xml")
games = soup_xml.find_all("item", attrs={"type": "boardgame"})

In [ ]:
boardgame_ids = df_bg_list["id"].astype(str).tolist()
batch_size = 5
for batch_num in range(math.ceil(len(xml_ids) / batch_size)):
    batch_ids = boardgame_ids[batch_num * batch_size : (batch_num + 1) * batch_size]
    batch_ids = [str(x) for x in batch_ids]
    bg_info_url = f"https://www.boardgamegeek.com/xmlapi2/thing?type=boardgame&stats=1&versions=1&ratingcomments=1&pagesize=10&page=1&id={','.join(batch_ids)}"
    bgg_response = requests.get(bg_info_url)
    soup_xml = BeautifulSoup(bgg_response.content, "xml")
    games = soup_xml.find_all("item", attrs={"type": "boardgame"})
    print(len(games))
    break

# Ratings

In [ ]:
def fetch_ratings_for_game(game_id, max_pages=1, delay=2):
    """
    Fetch all user ratings for a specific game_id from BGG.
    Returns a list of dicts with game_id, username, and rating.
    """
    page = 1
    all_ratings = []

    while True:
        url = f"https://boardgamegeek.com/xmlapi2/thing?id={game_id}&comments=1&page={page}"
        response = requests.get(url)

        while response.status_code == 202:
            print(
                f"[{game_id}] Waiting for BGG server to prepare comments... Retrying in 3s."
            )
            time.sleep(3)
            response = requests.get(url)

        if response.status_code != 200:
            print(f"[{game_id}] Error: {response.status_code}")
            break

        soup = BeautifulSoup(response.content, "xml")
        comments = soup.find_all("comment")

        if not comments:
            break  # No more comments/pages

        for comment in comments:
            username = comment.get("username")
            rating = comment.get("rating")

            if rating and rating != "N/A":
                all_ratings.append(
                    {
                        "game_id": int(game_id),
                        "username": username,
                        "rating": float(rating),
                    }
                )

        print(f"[{game_id}] Fetched page {page} with {len(comments)} comments.")
        page += 1

        if max_pages and page > max_pages:
            break

        time.sleep(delay)  # Be polite to the BGG server

    return all_ratings

In [ ]:
224517
rtngs_dict = fetch_ratings_for_game(game_id=224517)

In [ ]:
xml_ids = ["224517", "161936"]
xml_ids = ["224517"]
",".join(xml_ids)

In [ ]:
121//10

In [ ]:
# xml_ids = ["224517", "161936"]
xml_ids = ["161936"]
pg_num = 1
url = f"https://www.boardgamegeek.com/xmlapi2/thing?type=boardgame&ratingcomments=1&pagesize=100&page={pg_num}&id={','.join(xml_ids)}"
response = requests.get(url)
soup_xml = BeautifulSoup(response.content, "xml")

In [ ]:
item_list = []
items = soup_xml.find_all("item")
# for idx, item in enumerate(items):
#     item_list.append(extract_item(item, game_links[idx]))

In [ ]:
def game_data():
    xml_bs = "https://www.boardgamegeek.com/xmlapi2/thing?type=boardgame&stats=1&ratingcomments=1&page=1&pagesize=10&id="
    all_items = []
    for pg in range(1, 51):
        pg_items = []
        ct = 0
        soup_pg = browse_games(pg)
        pg_ids, pg_links = extract_game_ids(soup_pg)
        print(f"page number {pg} attempt number {ct}")
        while len(pg_items) != 100 and ct < 20:
            xml_fl = requests.get(f'{xml_bs}{",".join(pg_ids)}')
            soup_xml = BeautifulSoup(xml_fl.content, "xml")
            pg_items = extract_xml(soup_xml, pg_links)
            ct += 1
            if ct > 1:
                print(f"page number {pg} attempt number {ct}")
                print(len(pg_items))
        all_items += pg_items
    return all_items

def browse_games(page_num):
    bs_url = "https://boardgamegeek.com/browse/boardgame/page/"
    pg_url = f"{bs_url}{page_num}"
    if page_num <= 10:
        pg = requests.get(pg_url)
    else:
        pg = requests.get(pg_url, cookies)
    soup = BeautifulSoup(pg.content, "html.parser")
    return soup

def extract_game_ids(soup):
    bs_pg = "https://boardgamegeek.com/"
    all_games = soup.find_all("td", {"class": "collection_objectname"})
    game_ids = [x.find("a")["href"].split("/")[-2] for x in all_games]
    game_pages = [urljoin(bs_pg, x.find("a")["href"]) for x in all_games]
    return game_ids, game_pages

In [ ]:
df

In [ ]:
aa = soup.find("div", {"id": "maincontent"})("a")[0]["href"]

In [ ]:
df_ratings = pd.read_parquet("../../data/crawler/boardgame_ratings_1748670263.parquet")
df_ratings.shape

In [ ]:
# df_ratings = df_ratings.rename(columns={"game_id":"id"})
df_ratings["id"] = df_ratings["id"].astype(int)

In [ ]:
df_bg_data_2 = pd.read_parquet("../../data/crawler/boardgame_data_1749420286.parquet")

In [ ]:
df_bg_data_2.loc[df_bg_data_2["id"]==13,"numratings"]

In [ ]:
df_bg_data.loc[df_bg_data["id"]==13,"numratings"]

In [ ]:
df_ratings.to_parquet("../../data/crawler/boardgame_ratings_1748670264.parquet")

In [ ]:
boardgame_data = df_bg_data.loc[df_bg_data["numratings"] > 100].sort_values(
        by="numratings", ascending=False
    )
boardgame_ids = boardgame_data["id"].tolist()
len(boardgame_ids)
# 127594
# 21062

In [ ]:
boardgame_ratings= df_ratings.copy()
df_ratings_len = boardgame_ratings.copy()
df_ratings_len = df_ratings_len.drop(columns=["id"])
df_ratings_len = df_ratings_len.fillna("")
for col in df_ratings_len.columns:
    df_ratings_len[col] = df_ratings_len[col].apply(len)
df_ratings_pulled = pd.DataFrame(
    {
        "id": boardgame_ratings["id"].tolist(),
        "ratings_pulled": df_ratings_len.sum(axis=1).tolist(),
    }
)
boardgame_data = boardgame_data.merge(df_ratings_pulled, on="id", how="left")

In [ ]:

completed_ids = boardgame_data.loc[
    (boardgame_data["ratings_pulled"] - boardgame_data["numratings"])/(boardgame_data["numratings"])>=-0.01,
    "id",
].tolist()
boardgame_ids = list(set(boardgame_ids).difference(set(completed_ids)))
df_missing_ratings = boardgame_data.loc[
    (boardgame_data["ratings_pulled"] - boardgame_data["numratings"])/(boardgame_data["numratings"])<-0.01
]

In [ ]:
ratings_count_dict = pd.Series(
                df_missing_ratings["numratings"].values,
                index=df_missing_ratings["id"],
            ).to_dict()

In [ ]:
import math

In [ ]:
math.ceil(max(ratings_count_dict.values()) / 100)

In [ ]:
df_missing_ratings.shape

In [ ]:
134000/100

In [ ]:
boardgame_ratings = boardgame_ratings.loc[~(boardgame_ratings["id"].isin(df_missing_ratings["id"]))]

In [ ]:
boardgame_ratings.shape

In [ ]:
df_missing_ratings.shape

In [ ]:
df_ratings.shape

In [ ]:
df_missing_ratings.head()

In [ ]:
df_ratings_pulled.loc[df_ratings_pulled["id"]==266192]

In [ ]:
len(df_ratings.loc[df_ratings["id"]==13, "2.0"].iloc[0])

In [ ]:
df_ratings.shape

In [ ]:
df_ratings.head()

In [ ]:
for col in df_rats.columns[1:]:
    print(len(df_rats.loc[df_rats["id"]==13, col].iloc[0]))

In [ ]:
df_rats =pd.read_parquet("../../data/crawler/boardgame_ratings_1749423525.parquet")
df_rats.shape

In [ ]:
df_rats.head()

In [ ]:
'Dorowar' in df_ratings.loc[df_ratings["id"]==13, "10.0"].iloc[0]

In [ ]:
135689/100

In [ ]:
df = pd.read_parquet("../../data/crawler/boardgame_ratings_1749439709.parquet")
df.shape

In [ ]:
df.iloc[:1000].to_parquet("../../data/crawler/boardgame_ratings_1749439710.parquet")

In [ ]:
df = pd.read_csv("../../data/processed/processed_games_data_1749694079.csv", sep="|", escapechar="\\")
df.shape

In [ ]:
df.shape

In [ ]:
df.head()